# Logos Pretraining — Colab TPU v6e (Jax/Flax)

End-to-end pretraining of a **Logos** transformer on a **TPU v6e** runtime using **Jax + Flax + Optax + Orbax**.

**Architecture (Logos)**
- **KDA + Snapshot Retrieval** sub-quadratic attention with **MLA-compressed snapshots**
- **Local Sliding-Window Attention** every 4 layers (Samba-style)
- **Block Attention Residuals** (depth-wise softmax over completed blocks)
- **Three-section** Entry → Body (looped) → Exit
- **Mixture-of-Experts** body: shared + sparse experts with cross-loop expert diversity
- **Attention sink**, **QK norm**, **partial RoPE**

**Training pipeline**
- **Muon** (2D Linear weights via `optax.contrib.muon`) + **AdamW** for 1D params
- **WSD** schedule (warmup → stable → decay)
- **Rolling-N** checkpoints via **Orbax**
- **bf16** matmul precision, **jax.jit** compilation
- **Streaming** FineWeb-Edu corpus

> **Runtime:** set Colab to **TPU v6e** before running: `Runtime - Change runtime type - TPU v6e`.
> **Branch:** clone uses `jax-flax` branch with the Flax rewrite.


## 1. Install dependencies

We add Jax, Flax, Optax, Orbax and data deps.

In [ ]:
!pip install -U jax flax optax orbax-checkpoint datasets transformers tiktoken einops tqdm

In [ ]:
# Verify
import flax;  print('flax:', flax.__version__)
import optax; print('optax:', optax.__version__)
import orbax.checkpoint; print('orbax-checkpoint OK')


# 2. Test TPU Availability

In [ ]:
import jax
print('Jax version:', jax.__version__)
print('Devices:', jax.devices())
print('Backend:', jax.default_backend())
assert jax.default_backend() == 'tpu', 'Not running on TPU — switch runtime.'
print('TPU OK')

## 3. Get the Logos repo (jax-flax branch)

Replace the URL with your fork. Clones the `jax-flax` branch for the Flax rewrite.

In [ ]:
%cd /content/
import os, pathlib, subprocess
REPO_URL  = 'https://github.com/Rorical/Logos.git'
REPO_PATH = '/content/Logos'
BRANCH    = 'jax-flax'

if pathlib.Path(REPO_PATH).exists():
    print('Repo exists - fetching + resetting to', BRANCH)
    subprocess.check_call(['git', '-C', REPO_PATH, 'fetch', '--all', '--prune'])
    subprocess.check_call(['git', '-C', REPO_PATH, 'checkout', BRANCH])
    subprocess.check_call(['git', '-C', REPO_PATH, 'reset', '--hard', f'origin/{BRANCH}'])
else:
    subprocess.check_call(['git', 'clone', '-b', BRANCH, REPO_URL, REPO_PATH])

print('HEAD:')
subprocess.check_call(['git', '-C', REPO_PATH, 'log', '--oneline', '-1'])
%cd /content/Logos


## 4. Mount Google Drive (optional)

Checkpoints land in `MyDrive/logos-pretraining-jax/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import pathlib

CHECKPOINT_DIR = '/content/drive/MyDrive/logos-pretraining-jax'
pathlib.Path(CHECKPOINT_DIR).mkdir(parents=True, exist_ok=True)
print('Checkpoints:', CHECKPOINT_DIR)


## 5. Imports + Jax TPU info

In [ ]:
import sys, jax
sys.path.insert(0, '/content/Logos')

from scripts_jax.train import build_arg_parser, main
from models_jax import LogosConfig, LogosTransformer

print(f'Jax:      {jax.__version__}')
print(f'Devices:  {jax.devices()}')
print(f'Backend:  {jax.default_backend()}')


## 6. Build the training Namespace

All configuration passes through the argument parser into `scripts_jax/train.py`.

In [ ]:
parser = build_arg_parser()

cli = [
    '--model', 'logos',

    # Streaming corpus
    '--dataset',         'HuggingFaceFW/fineweb-edu',
    '--dataset-config',  'sample-10BT',
    '--text-column',     'text',
    '--tiktoken-encoding', 'cl100k_base',

    # Training horizon
    '--total-tokens', '10B',
    '--batch-size',   '8',
    '--max-length',   '4096',
    '--log-every',    '50',
    '--save-every',   '5000',
    '--keep-last-n',  '5',

    # Architecture
    '--d-model',          '1024',
    '--num-heads',        '16',
    '--head-dim',         '64',
    '--d-ff',             '2730',
    '--num-entry-layers', '2',
    '--num-body-layers',  '6',
    '--num-exit-layers',  '2',
    '--num-loops',        '3',
    '--dropout',          '0.0',
    '--norm-eps',         '1e-6',

    # KDA + retrieval
    '--chunk-size',          '128',
    '--conv-size',           '4',
    '--snapshot-interval',   '512',
    '--snapshot-latent-dim', '128',
    '--mem-top-k',           '16',
    '--mem-head-dim',        '64',

    # SWA
    '--swa-window', '256',
    '--swa-every',  '4',
    '--swa-offset', '3',

    # MoE
    '--use-moe',
    '--num-shared-experts',   '2',
    '--num-sparse-experts',   '32',
    '--top-k',                '4',
    '--expert-d-ff',          '512',
    '--moe-diversity-factor', '0.5',
    '--bias-update-rate',     '0.01',

    # RoPE
    '--rope-scaling-type',   'none',
    '--rope-scaling-factor', '1.0',

    # LM head
    '--lm-head-chunk-size', '512',

    # Optimizer — Muon + AdamW
    '--muon',
    '--lr',           '0.004',
    '--embed-lr',     '0.2',
    '--muon-lr',      '0.02',
    '--weight-decay', '0.1',
    '--grad-clip',    '1.0',

    # WSD schedule
    '--scheduler',    'wsd',
    '--warmup-steps', '3500',
    '--decay-steps',  '70000',

    # Output
    '--save-dir', CHECKPOINT_DIR,
    '--seed',     '42',
]

args = parser.parse_args(cli)
print('Run config:')
for k, v in sorted(vars(args).items()):
    print(f'  {k:<28} {v}')


## 7. Train

Single call to `main(args)`. Uses jax.jit compiled step, optax Muon+AdamW, Orbax checkpoints.

In [ ]:
main(args)


## 8. (Optional) Sample from the final model

Loads the final Orbax checkpoint and generates a continuation.

In [ ]:
import jax, jax.numpy as jnp
import orbax.checkpoint as ocp
from pathlib import Path
from models_jax import LogosConfig, LogosTransformer
from utils.tokenizer import TiktokenTokenizer

tok = TiktokenTokenizer(args.tiktoken_encoding)
cfg = build_config(args)  # if not defined, rebuild from args
model = LogosTransformer(cfg)

# Restore checkpoint
key = jax.random.PRNGKey(42)
dummy = jnp.ones((1, 64), dtype=jnp.int32)
variables = model.init(key, dummy, dummy, deterministic=True)

checkpointer = ocp.Checkpointer(ocp.StandardCheckpointHandler())
restored = checkpointer.restore(
    str(Path(CHECKPOINT_DIR) / 'final'),
    args=ocp.args.StandardRestore(variables),
)

params = restored['params']
moe_vars = {k: v for k, v in restored.items() if k != 'params'}

# Generate
prompt = 'The history of artificial intelligence began'
input_ids = jnp.array([tok.encode(prompt)], dtype=jnp.int32)

for _ in range(200):
    logits = model.apply(
        {'params': params, **moe_vars},
        input_ids, deterministic=True,
    )['logits']
    next_tok = jnp.argmax(logits[:, -1], axis=-1)
    input_ids = jnp.concatenate([input_ids, next_tok[:, None]], axis=1)

print(tok.decode(input_ids[0].tolist()))


## 9. (Optional) Plot the loss curve

In [ ]:
import json, matplotlib.pyplot as plt

with (Path(CHECKPOINT_DIR) / 'history.json').open() as f:
    hist = json.load(f)

events = hist['history']
steps = [h['step'] for h in events]
losses = [h['loss'] for h in events]

plt.figure(figsize=(10, 5))
plt.plot(steps, losses, '-', linewidth=2)
plt.xlabel('step'); plt.ylabel('loss')
plt.grid(alpha=0.3); plt.title('Logos pretraining (Jax/Flax)')
plt.show()
